# Local Support Assembly Demo

This notebook demonstrates exact support-closure local assembly. Starting from a non-overlapping element partition, the support patch is defined by DoF support rather than by a geometric overlap layer.

In [ ]:
import sys
from pathlib import Path

candidate_build_dirs = [
    Path.cwd() / "build-local",
    Path.cwd() / "build",
    Path.cwd() / "NGS-myassembling" / "build-local",
    Path.cwd() / "NGS-myassembling" / "build",
]
for build_dir in candidate_build_dirs:
    if build_dir.exists():
        sys.path.insert(0, str(build_dir))
        break

import numpy as np
from ngsolve import *
from ngsolve.webgui import Draw
from netgen.geom2d import unit_square

import myassembling
print(myassembling)
print(myassembling.__file__)
print(dir(myassembling))

SetNumThreads(1)

## Mesh, Space, And Core Element Partition

In a production workflow, `partition[i]` would typically come from METIS and contain the global volume element numbers in the non-overlapping core subdomain. Here we split the mesh into three groups by element-center x coordinate.

In [ ]:
mesh = Mesh(unit_square.GenerateMesh(maxh=0.12, quad_dominated=False))
fes = H1(mesh, order=1, dirichlet="left|right|bottom|top")
bfi = myassembling.MyLaplace(CoefficientFunction(1.0))

partition = [[], [], []]
for el in mesh.Elements(VOL):
    pts = [mesh[v].point for v in el.vertices]
    cx = sum(p[0] for p in pts) / len(pts)
    part = min(2, int(3 * cx))
    partition[part].append(el.nr)

print("ne =", mesh.ne, "ndof =", fes.ndof)
print("core element counts =", [len(p) for p in partition])

## Visualize The Core Partition

The discontinuous order-zero grid function stores one partition id per element.

In [ ]:
l2 = L2(mesh, order=0)
omega_id = GridFunction(l2, name="Omega_i")
for i, elements in enumerate(partition):
    for elnr in elements:
        ei = ElementId(VOL, elnr)
        omega_id.vec[l2.GetDofNrs(ei)[0]] = i + 1

Draw(omega_id, mesh, "core element partition")

## Assemble Local Support Matrices

For each core element set, the support closure is:

- `core_dofs`: all global DoFs appearing on `core_elements`
- `support_elements`: all elements whose DoF list intersects `core_dofs`
- `support_dofs`: all global DoFs appearing on `support_elements`
- `core_in_support`: local indices of `core_dofs` inside `support_dofs`

The local matrix is assembled only over `support_elements` using `support_dofs` as local numbering.

In [ ]:
patches = myassembling.MyAssembleLocalSupportMatrices(fes, bfi, partition)

for i, patch in enumerate(patches):
    print(f"Omega_{i}:")
    print("  core_elements   =", len(patch.core_elements))
    print("  support_elements=", len(patch.support_elements))
    print("  core_dofs       =", len(patch.core_dofs))
    print("  support_dofs    =", len(patch.support_dofs))
    print("  core_in_support =", list(patch.core_in_support))
    print("  matrix shape    =", (patch.mat.height, patch.mat.width))

    for k, g in enumerate(patch.core_dofs):
        assert patch.support_dofs[patch.core_in_support[k]] == g

## Visualize One Support Patch

Value `1` marks core elements, value `2` marks support elements that are not in the core, and value `0` marks all other elements.

In [ ]:
for patch_id in range(len(patches)):
    patch = patches[patch_id]

    patch_view = GridFunction(l2, name="support_patch_view")
    patch_view.vec[:] = 0
    for elnr in patch.support_elements:
        patch_view.vec[l2.GetDofNrs(ElementId(VOL, elnr))[0]] = 2
    for elnr in patch.core_elements:
        patch_view.vec[l2.GetDofNrs(ElementId(VOL, elnr))[0]] = 1

    Draw(patch_view, mesh, f"Omega_{patch_id} core and support closure")

## Verify Against Global Assembly

The intended correctness condition is only the core-core block:

`A_support[core_in_support, core_in_support] == A_global[core_dofs, core_dofs]`

We do not compare `A_support` against `A_global[support_dofs, support_dofs]`, because the support matrix is assembled only from elements touching `core_dofs`.

In [ ]:
A_global = myassembling.MyAssembleMatrix(fes, bfi)

def dense_submatrix(mat, rows, cols):
    return np.array([[mat[i, j] for j in cols] for i in rows], dtype=float)

for i, patch in enumerate(patches):
    A_support = np.array(patch.mat.ToDense(), dtype=float)
    ids = list(patch.core_in_support)
    A_core_from_support = A_support[np.ix_(ids, ids)]
    A_core_from_global = dense_submatrix(A_global, patch.core_dofs, patch.core_dofs)
    err = np.linalg.norm(A_core_from_support - A_core_from_global, ord=np.inf)
    print(f"Omega_{i}: ||A_support[core,core] - A_global[core,core]||_inf = {err:.3e}")
    assert err < 1e-12